# Image: folder to prediction

Five steps, one notebook, no other page:

1. `datasets.upload()` — your folder becomes a dataset.
2. `train.start()` — the platform's own training job.
3. `evaluate.start()` — the recorded score.
4. `inference.predict()` — one image through the model you just trained.
5. `runs.log()` — what happened, kept.

Every call is `orinth`. Nothing here is a notebook-only shortcut: the dataset
appears on `/datasets`, the run on `/training`, the model in the catalog.

In [ ]:
import orinth
from pathlib import Path

# Your images: one subdirectory per class. The structure *is* the labelling.
SOURCE = Path("~/Downloads/my-images").expanduser()
print(SOURCE, "exists:", SOURCE.exists())

No folder there? The next cell borrows sixty images from a project dataset and
lays them out the way yours would be, so the rest of the notebook runs. Point
`SOURCE` at your own and re-run from here.

In [ ]:
import shutil
import tempfile

import polars as pl

if not SOURCE.exists():
    donor = max(
        (d for d in orinth.datasets.list() if d.task_type == "classification" and d.readiness.trainable),
        key=lambda d: d.total_items,
    )
    SOURCE = Path(tempfile.mkdtemp()) / "example-images"
    # Per class, not the first N rows: `limit=` is a head, and items come back in
    # directory order, so `load(..., limit=90)` on a six-class set is ninety
    # images of one class — which is a folder the prep agent will (correctly)
    # refuse to label.
    borrowed = orinth.datasets.load(donor.id, "train", limit=1200)
    for label in borrowed["label"].unique().to_list()[:3]:
        rows = borrowed.filter(pl.col("label") == label).head(30)
        folder = SOURCE / str(label)
        folder.mkdir(parents=True, exist_ok=True)
        for row in rows.iter_rows(named=True):
            shutil.copy2(row["path"], folder / Path(row["path"]).name)
    print("borrowed from", donor.id)

classes = sorted(p.name for p in SOURCE.iterdir() if p.is_dir())
print(SOURCE, "|", classes)

## 1 · Upload

In [ ]:
dataset = orinth.datasets.upload(SOURCE, name="image pipeline")
print(dataset.id, "·", dataset.task_type, "·", dataset.format)
print(dataset.labels)
print(dataset.splits, "|", dataset.readiness.state)

## 2 · Train

`options()` carries each entry's defaults, so the numbers below start from the
catalog rather than from guesswork. Epochs are low so this finishes while you
watch — raise them for anything real.

In [ ]:
option = next(entry for entry in orinth.train.options("classification") if entry["runnable"])
print("training with", option["id"])

run = orinth.train.start(
    dataset.id,
    model_option_id=option["id"],
    task_type="classification",
    name="image pipeline",
    epochs=3,
    image_size=160,
    batch_size=8,
)
final = orinth.train.wait_for(run.id, on_progress=lambda r: print(f"{r.percent:5.1f}%  {r.step}"))
print("\n", final.status, "->", final.model_id)

## 3 · Evaluate

Recorded, on `/testing`, against the split the platform holds out — not against
the rows the model just trained on.

In [ ]:
key = next(
    entry["key"]
    for entry in orinth.evaluate.datasets()
    if entry["key"].startswith(f"dataset:{dataset.id}:")
)
def overall_of(metrics):
    # Metrics are shaped per task — `image` for vision, `text_classification`
    # for text — so the headline block is *found* rather than guessed at by key.
    for value in metrics.values():
        if isinstance(value, dict) and isinstance(value.get("overall"), dict):
            return value["overall"]
    return {}


job = orinth.evaluate.start(final.model_id, key)
overall = overall_of(job.metrics)
print(job.status, "|", {k: round(v, 3) for k, v in overall.items()})

## 4 · Predict

In [ ]:
sample = orinth.datasets.paths(dataset.id, "test")[0]
prediction = orinth.inference.predict(final.model_id, sample)
print(sample.name, "->", prediction.label)
print({k: round(v, 3) for k, v in sorted(prediction.scores.items(), key=lambda kv: -kv[1])[:3]})
print("overlay:", prediction.overlay_url)

## 5 · Keep it

One run holds the whole pipeline: which dataset, which model, what it scored.
That is what makes the second attempt comparable to the first.

In [ ]:
with orinth.runs.start(
    "image pipeline",
    params={"dataset": dataset.id, "model": final.model_id, "option": option["id"]},
    dataset_id=dataset.id,
) as pipeline:
    for key_name, value in overall.items():
        if isinstance(value, (int, float)):
            pipeline.log(**{key_name: float(value)})
    pipeline.log_text("classes", ", ".join(dataset.labels))